# Week 4 · Hands-on: Multi-Agent Coordination with CAMEL and Swarm

This is where coordination theory becomes a running system.

In Parts 1 and 2, you studied the design space of multi-agent coordination — centralized vs. decentralized architectures, communication protocols, consensus, role specialization, dynamic task allocation, and game-theoretic dynamics. In Reading 1, you read the CAMEL paper that introduced **inception prompting** as a mechanism for structured two-agent collaboration.

In this exercise, you'll build two things:

1. **Stage 1: RolePlaying** — A two-agent pair that collaborates on a task through structured turn-taking with inception prompting
2. **Stage 2: Swarm** — A triage agent that routes work to specialist agents through explicit handoff functions (OpenAI Swarm)

**Estimated time:** ~45 minutes

**Reference docs:**
- [Creating Your First Agent](https://docs.camel-ai.org/cookbooks/basic_concepts/create_your_first_agent)
- [Creating Your First Agent Society](https://docs.camel-ai.org/cookbooks/basic_concepts/create_your_first_agents_society)
- [Workforce Module](https://docs.camel-ai.org/key_modules/workforce)


## Setup — Install CAMEL and Configure API Key


In [ ]:
# mcp<2: camel-ai 0.2.90 imports FastMCP, which mcp 2.x removed
!pip install --quiet 'camel-ai[model_platforms]' 'mcp<2'

**Note:** Colab may print a red `ERROR: pip's dependency resolver...` message that mentions packages like google-genai or pyiceberg. That's harmless for this notebook, so keep going. If Colab asks you to restart the session, restart it and continue from the next cell.


### Set your OpenAI API key

**Option A — Colab Secrets (recommended):**
1. Click the 🔑 (key) icon in the left sidebar
2. Click **+ Add new secret**
3. Name: `OPENAI_API_KEY`, Value: paste your key
4. Toggle the notebook access switch ON

Then run the cell below. It uses the Colab secret if one is set.

**Option B — Direct entry:** skip the secret and run the cell below. It asks you to paste your key.


In [ ]:
import os

# Option A: Colab Secrets (key icon in the left sidebar, name OPENAI_API_KEY, notebook access ON)
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception as e:  # secret missing, access off, or not running in Colab
    print(f"Colab secret not used ({type(e).__name__}); falling back to manual entry.")

# Option B: Manual entry
if not os.environ.get("OPENAI_API_KEY"):
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("✅ API key configured")

### Configure the model

CAMEL uses `ModelFactory` to create model instances. We'll use `gpt-4o-mini` — fast, cheap, and sufficient for this exercise.


In [4]:
from camel.models import ModelFactory
from camel.types import ModelPlatformType, ModelType
from camel.configs import ChatGPTConfig

model = ModelFactory.create(
    model_platform=ModelPlatformType.OPENAI,
    model_type=ModelType.GPT_4O_MINI,
    model_config_dict=ChatGPTConfig(temperature=0.0).as_dict(),
)

print(f"✅ Model ready: {ModelType.GPT_4O_MINI}")


✅ Model ready: gpt-4o-mini


---

# Stage 1: RolePlaying — Structured Two-Agent Collaboration

Two agents receive roles and collaborate on a task through structured turn-taking. CAMEL's **inception prompting** enforces role boundaries — agents can't flip roles, must provide structured solutions, and terminate when the task is done.

| What you're building | Coordination concept |
|---|---|
| Two agents with assigned roles | Role specialization (Part 2) |
| Turn-taking communication loop | Direct messaging protocol (Part 1) |
| Inception prompting preventing role-flipping | Consensus on responsibilities (Part 1) |
| `CAMEL_TASK_DONE` termination | Cooperative termination (Part 2) |
| `with_task_specify` flag | Task decomposition before execution (Part 1) |


## Step 1.1 — Configure the RolePlaying session

We set up two agents — a **Product Manager** (the instruction sender) and a **Senior Software Architect** (the instruction executor) — and give them a task. The `with_task_specify=True` flag activates a third agent that refines the high-level task into something more concrete before the conversation begins.

> 💡 **Try your own roles and task.** The example below uses a system design scenario, but you can change the roles and task to anything you find interesting. The coordination mechanics work the same regardless of domain.


In [ ]:
from camel.societies import RolePlaying

TASK = "Design a microservices architecture for a real-time fraud detection system"

role_play_session = RolePlaying(
    assistant_role_name="Senior Software Architect",
    assistant_agent_kwargs=dict(model=model),
    user_role_name="Product Manager",
    user_agent_kwargs=dict(model=model),
    task_prompt=TASK,
    with_task_specify=True,
    task_specify_agent_kwargs=dict(model=model),
)

print("✅ RolePlaying session created")


## Step 1.2 — Inspect the inception prompts

Before running the conversation, let's look at what CAMEL generated. The system messages contain the **inception prompts** — the carefully crafted instructions that prevent role-flipping, enforce structured responses, and keep agents on task.

This is the mechanism you read about in the CAMEL paper (Li et al., 2023, §3).


In [ ]:
print("=" * 60)
print("ORIGINAL TASK:")
print("=" * 60)
print(TASK)  # CAMEL replaces task_prompt with the specified task, so keep our own copy

print("\n" + "=" * 60)
print("SPECIFIED TASK (refined by task-specify agent):")
print("=" * 60)
print(role_play_session.specified_task_prompt)

print("\n" + "=" * 60)
print("AI USER (Product Manager) SYSTEM MESSAGE:")
print("=" * 60)
print(role_play_session.user_sys_msg.content[:500])
print("..." if len(role_play_session.user_sys_msg.content) > 500 else "")

print("\n" + "=" * 60)
print("AI ASSISTANT (Software Architect) SYSTEM MESSAGE:")
print("=" * 60)
print(role_play_session.assistant_sys_msg.content[:500])
print("..." if len(role_play_session.assistant_sys_msg.content) > 500 else "")


## Step 1.3 — Run the conversation

Now we run the turn-taking loop. Watch how:
- The **Product Manager** provides instructions and requirements
- The **Software Architect** responds with structured solutions
- Each turn builds on the previous one
- The conversation terminates when the task is done (`CAMEL_TASK_DONE`)


In [7]:
input_msg = role_play_session.init_chat()
n = 0
max_turns = 10

while n < max_turns:
    n += 1
    assistant_response, user_response = role_play_session.step(input_msg)

    print(f"\n{'━' * 60}")
    print(f"  TURN {n}")
    print(f"{'━' * 60}")

    # Print the user (Product Manager) message
    user_content = user_response.msg.content
    print(f"\n🟦 [Product Manager]:")
    print(user_content[:400])
    if len(user_content) > 400:
        print(f"  ... ({len(user_content)} chars total)")

    # Print the assistant (Architect) message
    asst_content = assistant_response.msg.content
    print(f"\n🟩 [Software Architect]:")
    print(asst_content[:400])
    if len(asst_content) > 400:
        print(f"  ... ({len(asst_content)} chars total)")

    # Check termination conditions
    if assistant_response.terminated:
        print(f"\n🛑 Assistant terminated: {assistant_response.info.get('termination_reasons', 'unknown')}")
        break
    if user_response.terminated:
        print(f"\n🛑 User terminated: {user_response.info.get('termination_reasons', 'unknown')}")
        break
    if "CAMEL_TASK_DONE" in user_response.msg.content:
        print("\n✅ Task completed! (CAMEL_TASK_DONE detected)")
        break

    input_msg = assistant_response.msg

print(f"\n{'=' * 60}")
print(f"Session ended after {n} turn(s)")
print(f"{'=' * 60}")



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TURN 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🟦 [Product Manager]:
Instruction: Define the key microservices needed for the real-time fraud detection system.  
Input: None

🟩 [Software Architect]:
Solution: To design a microservices architecture for a real-time fraud detection system, we need to define several key microservices that will work together to achieve the desired functionality. Here are the essential microservices:

1. **Transaction Analysis Service**:
   - **Purpose**: This service is responsible for analyzing incoming transactions in real-time to identify potentially fraudulent
  ... (3313 chars total)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TURN 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🟦 [Product Manager]:
Instruction: Outline the communication methods between the defined microservices.  
Input: None

🟩 [Software Architect]:
Solution: In a 

## 🔍 Stage 1 Reflection

Before moving to Stage 2, answer these questions (discuss with your neighbor or jot down notes):

1. **Inception prompts:** Read the system messages above. What specific rules prevent the agents from flipping roles? How does this compare to CrewAI's `role`/`goal`/`backstory` approach from Week 1?

2. **Task specification:** Compare the original task to the specified version. How did the task-specify agent refine it? Is this a form of task decomposition?

3. **Communication protocol:** This is a direct messaging, turn-taking protocol. Where does it fall on the centralized–decentralized spectrum from Part 1? Is there an orchestrator?

4. **Convergence:** How many turns did the agents need? Did they converge on a solution, or would they keep going indefinitely without the turn limit?

5. **Cooperative dynamics:** Both agents are working toward the same goal. How would the conversation change if they had competing objectives (e.g., a VC negotiating with a startup founder)?


---

# Stage 2: Swarm — Decentralized Multi-Agent Coordination via Handoffs

Now we move from CAMEL's peer-to-peer collaboration to a **decentralized handoff-based architecture** using [OpenAI Swarm](https://github.com/openai/swarm) — a lightweight, educational framework for multi-agent orchestration.

Where CAMEL's Workforce uses a **centralized coordinator** that decomposes tasks and assigns them to workers behind the scenes, Swarm makes coordination **explicit and transparent** through two primitives:

- **Agents** — each with specialized instructions and tools
- **Handoffs** — functions that transfer control from one agent to another

This gives you full visibility into *how* and *when* coordination decisions happen — no hidden planning loops or opaque task queues.

| What you're building | Coordination concept |
|---|---|
| Triage agent routing work | Centralized dispatch (Part 1) |
| Handoff functions between agents | Explicit communication protocol (Part 1) |
| Specialized agents with distinct instructions | Role specialization / division of labor (Part 2) |
| Agent returning results to triage | Cooperative multi-agent scenario (Part 2) |
| No hidden orchestration layer | Transparency vs. abstraction tradeoff (Part 1) |


## Step 2.0 — Install Swarm


In [ ]:
# --no-deps: Swarm only needs openai + pydantic (already installed). Its declared deps pull
# instructor 1.3.2, which downgrades openai mid-session and breaks client.run().
!pip install --quiet --no-deps git+https://github.com/openai/swarm.git

## Step 2.1 — Create specialized agents

Each Swarm `Agent` gets a `name`, `instructions` (the system prompt), and a `model`. Unlike CAMEL's Workforce — where a hidden coordinator reads description strings to assign work — Swarm agents are routed via **explicit handoff functions** that you define.

> 💡 **Notice the difference:** In CAMEL Workforce, the coordinator *decides* who handles what using LLM reasoning over description strings. In Swarm, *you* define the routing logic as Python functions. This is the centralized-vs-decentralized tradeoff from Part 1 made concrete.


In [9]:
from swarm import Swarm, Agent

MODEL = "gpt-4o-mini"  # same model as Stage 1 — fast and cheap

researcher = Agent(
    name="Market Researcher",
    model=MODEL,
    instructions="""You are a Market Research Analyst. You specialize in
    analyzing market trends, competitor landscapes, and customer segments.
    You provide data-driven insights with specific metrics and sources.
    Always structure your analysis with clear sections and bullet points.
    When you finish your analysis, be thorough but concise.""",
)

strategist = Agent(
    name="Business Strategist",
    model=MODEL,
    instructions="""You are a Business Strategist. You specialize in
    developing go-to-market strategies, pricing models, and growth plans.
    You create actionable strategic recommendations with timelines,
    resource requirements, and success metrics.
    When you finish your strategy, be thorough but concise.""",
)

writer = Agent(
    name="Technical Writer",
    model=MODEL,
    instructions="""You are a Technical Writer. You specialize in
    synthesizing research and strategy into clear, compelling documents.
    You produce executive summaries, reports, and presentations
    suitable for C-level audiences.
    When you finish writing, be thorough but concise.""",
)

print("✅ Three specialist agents created: Researcher, Strategist, Writer")


✅ Three specialist agents created: Researcher, Strategist, Writer


## Step 2.2 — Build the triage agent with handoff functions

The **triage agent** is Swarm's equivalent of CAMEL's coordinator. But instead of using an LLM to plan and assign subtasks behind the scenes, the triage agent has **handoff functions** — regular Python functions that return another agent.

When the triage agent decides it needs market research, it calls `transfer_to_researcher()`, which returns the `researcher` agent. Swarm then switches execution to that agent. This is **the** core Swarm primitive.

> 💡 **The handoff functions are tools.** Swarm converts them to JSON schemas via their docstrings and passes them to the Chat Completions API as `tools`. The LLM decides which function to call based on the docstring descriptions — these serve the same role as CAMEL's worker description strings.


In [10]:
# Handoff functions — the coordination mechanism
def transfer_to_researcher():
    """Transfer to the Market Researcher for market sizing, competitor
    analysis, industry trends, and data-driven insights."""
    return researcher

def transfer_to_strategist():
    """Transfer to the Business Strategist for go-to-market strategy,
    pricing models, partnership opportunities, and growth roadmaps."""
    return strategist

def transfer_to_writer():
    """Transfer to the Technical Writer for executive summaries,
    investor presentations, and polished reports."""
    return writer

# The triage agent — Swarm's version of CAMEL's coordinator
triage_agent = Agent(
    name="Triage Coordinator",
    model=MODEL,
    instructions="""You are a project coordinator. Your job is to understand
    the user's request and transfer them to the right specialist:
    - Market Researcher: for market data, sizing, competitors, trends
    - Business Strategist: for GTM strategy, pricing, growth plans
    - Technical Writer: for executive summaries, reports, presentations

    Analyze the request and immediately transfer to the best specialist.
    Do not try to answer yourself.""",
    functions=[transfer_to_researcher, transfer_to_strategist, transfer_to_writer],
)

print("✅ Triage coordinator created with 3 handoff functions")
print("   Handoffs:")
print("   • transfer_to_researcher() → Market Researcher")
print("   • transfer_to_strategist() → Business Strategist")
print("   • transfer_to_writer()     → Technical Writer")


✅ Triage coordinator created with 3 handoff functions
   Handoffs:
   • transfer_to_researcher() → Market Researcher
   • transfer_to_strategist() → Business Strategist
   • transfer_to_writer()     → Technical Writer


## Step 2.3 — Run tasks through the Swarm

Now we send each subtask to the triage agent. Watch what happens:
1. The triage agent reads the request
2. It calls the appropriate handoff function (visible in the output)
3. Swarm switches execution to the specialist agent
4. The specialist produces the result

Each call completes in **seconds**, not minutes — because there's no hidden planning loop, no internal coordinator↔worker conversation, no async task queue. Just one LLM call for routing + one LLM call for execution.


In [11]:
import time

client = Swarm()

tasks = [
    "Estimate the market size and 5-year growth projections for AI-powered "
    "supply chain optimization platforms targeting mid-size e-commerce "
    "companies (annual revenue $10M–$500M).",

    "Identify the top 3 competitors in AI supply chain optimization for "
    "mid-size e-commerce and analyze each one's key weaknesses.",

    "Recommend a go-to-market strategy with 3 pricing tiers for an "
    "AI supply chain optimization platform targeting mid-size e-commerce.",

    "Write a 200-word executive summary for a seed-round investor deck "
    "about an AI supply chain optimization platform for mid-size e-commerce.",
]

results = {}

for i, task in enumerate(tasks, 1):
    print(f"\n{'━' * 60}")
    print(f"  TASK {i}")
    print(f"{'━' * 60}")
    print(f"📋 {task[:80]}...")

    start = time.time()
    response = client.run(
        agent=triage_agent,
        messages=[{"role": "user", "content": task}],
    )
    elapsed = time.time() - start

    # Who handled it?
    handler = response.agent.name
    content = response.messages[-1]["content"]
    results[f"task_{i}"] = {"handler": handler, "content": content}

    print(f"\n🔀 Routed to: {handler}")
    print(f"⏱️  Completed in {elapsed:.1f}s")
    print(f"\n{content[:500]}")
    if len(content) > 500:
        print(f"  ... ({len(content)} chars total)")

print(f"\n{'=' * 60}")
print("✅ All 4 tasks complete")
print(f"{'=' * 60}")



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TASK 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 Estimate the market size and 5-year growth projections for AI-powered supply cha...

🔀 Routed to: Market Researcher
⏱️  Completed in 13.7s

### Market Size Estimation for AI-Powered Supply Chain Optimization Platforms

#### **1. Current Market Size**
- **Target Industry**: E-commerce
- **Target Segment**: Mid-size companies with annual revenue of $10M–$500M
- **Estimated Total Number of Mid-Size E-commerce Companies**: 
  - According to the U.S. Census Bureau, there are approximately 100,000 e-commerce businesses in the United States.
  - Estimates indicate that around 10-15% fall within the mid-size category—yielding approximately 
  ... (2742 chars total)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TASK 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 Identify the top 3 competitors in AI supply chain optimization for

## Step 2.4 — Examine the routing decisions


In [12]:
print("=" * 60)
print("  ROUTING SUMMARY")
print("=" * 60)
for task_id, result in results.items():
    print(f"\n{task_id}: → {result['handler']}")
    print(f"  Output: {result['content'][:150]}...")


  ROUTING SUMMARY

task_1: → Market Researcher
  Output: ### Market Size Estimation for AI-Powered Supply Chain Optimization Platforms

#### **1. Current Market Size**
- **Target Industry**: E-commerce
- **T...

task_2: → Market Researcher
  Output: ### Competitor Analysis in AI Supply Chain Optimization for Mid-Size E-Commerce

#### 1. **Blue Yonder (formerly JDA Software)**
   - **Overview**: Bl...

task_3: → Business Strategist
  Output: ### Go-to-Market Strategy for AI Supply Chain Optimization Platform

#### Target Audience
- Mid-size e-commerce businesses looking to enhance operatio...

task_4: → Technical Writer
  Output: **Executive Summary**  \n\nIn the rapidly evolving landscape of e-commerce, mid-size enterprises face significant challenges in supply chain managemen...


## Step 2.5 — Multi-hop: chaining agents in a single conversation

Swarm also supports **multi-turn conversations** where the active agent changes mid-conversation. Here we give the specialist agents the ability to hand off *back* to the triage agent or to each other — creating a chain.

This demonstrates a pattern that CAMEL's Workforce can't easily do: **lateral handoffs** between peers, not just top-down delegation.


In [13]:
# Give the researcher a handoff to the strategist for follow-up
def transfer_to_strategist_for_followup():
    """After completing research, transfer to the Business Strategist
    to develop strategy based on the research findings."""
    return strategist

# Create a research agent that can chain to strategy
researcher_chained = Agent(
    name="Market Researcher",
    model=MODEL,
    instructions="""You are a Market Research Analyst. Analyze the market
    data requested. After completing your analysis, transfer to the
    Business Strategist to develop a strategy based on your findings.""",
    functions=[transfer_to_strategist_for_followup],
)

print("📋 Running chained task: Research → Strategy\n")

start = time.time()
response = client.run(
    agent=researcher_chained,
    messages=[{"role": "user", "content":
        "Analyze the competitive landscape for AI supply chain platforms "
        "targeting mid-size e-commerce, then develop a differentiation strategy."
    }],
)
elapsed = time.time() - start

print(f"🔀 Final agent: {response.agent.name}")
print(f"⏱️  Completed in {elapsed:.1f}s")
print(f"\n{response.messages[-1]['content'][:600]}")
if len(response.messages[-1]['content']) > 600:
    print(f"  ... ({len(response.messages[-1]['content'])} chars total)")


📋 Running chained task: Research → Strategy

🔀 Final agent: Business Strategist
⏱️  Completed in 30.9s

### Competitive Landscape Analysis for AI Supply Chain Platforms in Mid-size E-commerce

1. **Key Players**:
   - **Established Market Leaders**: Companies like SAP, Oracle, and IBM offer comprehensive, high-end AI supply chain solutions but may be too complex and expensive for mid-size e-commerce businesses.
   - **Niche Providers**: These include platforms like ShipBob, Flexport, and Easyship that focus more on logistics and fulfillment, but some are starting to incorporate AI capabilities.
   - **Emerging Startups**: Newer entrants such as Locus and Zencargo provide innovative solutions tai
  ... (4631 chars total)


## 🔍 Stage 2 Reflection

1. **Coordination mechanism:** CAMEL's Workforce uses a hidden coordinator agent that decomposes tasks and assigns them via LLM reasoning. Swarm uses explicit handoff functions that *you* define. What are the tradeoffs? When would you prefer each approach?

2. **Routing accuracy:** Did the triage agent route each task to the right specialist? The routing decision was based on the handoff function **docstrings** — these serve the same role as CAMEL's worker description strings. What happens if you make the docstrings vague?

3. **Speed:** Each Swarm task completed in seconds. Count the LLM calls: Swarm makes ~2 calls per task (triage + specialist). A coordinator-style system such as CAMEL Workforce adds planning, assignment and synthesis calls on top. How would that change latency and cost?

4. **Centralized vs. decentralized:** Where does Swarm's handoff pattern fall on the centralized–decentralized spectrum from Part 1? The triage agent is a *lightweight* central dispatcher, but the handoff functions are explicit code — not LLM-driven planning. Is this more or less centralized than CAMEL Workforce?

5. **Multi-hop chaining (Step 2.5):** The researcher handed off to the strategist laterally. CAMEL's Workforce only supports top-down delegation (coordinator → worker). When would lateral handoffs be more useful than hierarchical delegation?

6. **Compare to Stage 1:** RolePlaying had two peers collaborating through turn-taking. Swarm has a triage agent dispatching to specialists. How would you combine both patterns — e.g., a Swarm triage agent that hands off to a CAMEL RolePlaying pair for complex subtasks?


---

# Stretch Goals

If you finish early, try one of these extensions.


## Stretch A — Add a Critic to RolePlaying

CAMEL's RolePlaying supports an optional **critic agent** that reviews the assistant's responses for quality control. This adds a third coordination role — quality assurance — to the two-agent conversation. Note: the critic chooses among *options*. With the default single completion (n=1) it only ever sees Option 1, so it adds a call per message without changing anything. To give it a real choice, build a second model with `ChatGPTConfig(temperature=0.7, n=3).as_dict()` and pass it in `assistant_agent_kwargs` and `user_agent_kwargs` (keep `critic_kwargs=dict(model=model)`). That roughly triples output tokens.


In [14]:
# Re-create the session with a critic in the loop
critic_session = RolePlaying(
    assistant_role_name="Senior Software Architect",
    assistant_agent_kwargs=dict(model=model),
    user_role_name="Product Manager",
    user_agent_kwargs=dict(model=model),
    task_prompt="Design an API rate limiting strategy for a multi-tenant SaaS platform",
    with_task_specify=True,
    task_specify_agent_kwargs=dict(model=model),
    with_critic_in_the_loop=True,
    critic_role_name="Engineering Director",
    critic_criteria="Evaluate for scalability, security, and operational complexity",
    critic_kwargs=dict(model=model),
)

# Run a few turns and observe how the critic changes the dynamic
input_msg = critic_session.init_chat()

for turn in range(5):
    assistant_response, user_response = critic_session.step(input_msg)
    print(f"\nTurn {turn + 1}:")
    print(f"  User: {user_response.msg.content[:200]}...")
    print(f"  Asst: {assistant_response.msg.content[:200]}...")

    if assistant_response.terminated or user_response.terminated:
        break
    if "CAMEL_TASK_DONE" in user_response.msg.content:
        break
    input_msg = assistant_response.msg

# How did the critic change the conversation? Was quality higher?



Turn 1:
  User: Instruction: Define the request limits for each user role (free, pro, enterprise) in the tiered API rate limiting strategy.  
Input: None...
  Asst: Solution: 

To define the request limits for each user role in the tiered API rate limiting strategy for a multi-tenant SaaS platform, we can establish distinct thresholds based on the expected usage ...


/usr/local/lib/python3.12/dist-packages/camel/memories/base.py:176: UserWarning: Chat history window size limit (6) reached. Some earlier messages will not be included in the context. Consider increasing window_size if you need a longer context.
  return self.get_context_creator().create_context(self.retrieve())



Turn 2:
  User: Instruction: Design a mechanism for dynamic adjustments of rate limits based on usage patterns.  
Input: None...
  Asst: Solution:

To design a mechanism for dynamic adjustments of rate limits based on usage patterns, we can implement a system that monitors user behavior and adjusts limits in real-time or near real-time...

Turn 3:
  User: Instruction: Implement a real-time feedback mechanism to inform users about their current usage and limits.  
Input: None...
  Asst: Solution:

To implement a real-time feedback mechanism that informs users about their current usage and limits, we can utilize HTTP response headers in our API responses. This approach allows users to...

Turn 4:
  User: Instruction: Ensure seamless integration of the rate limiting strategy with existing authentication mechanisms.  
Input: None...
  Asst: Solution:

To ensure seamless integration of the rate limiting strategy with existing authentication mechanisms in a multi-tenant SaaS platform, we ne

## Stretch B — Use Context Variables for Shared State

Swarm supports `context_variables` — a shared dictionary that persists across agent handoffs. This lets agents pass structured data (not just conversation history) to each other. Try using context variables to pass the researcher's findings to the strategist as structured data.

```python
from swarm.types import Result  # swarm/__init__ exports only Swarm, Agent, Response

def research_and_handoff(context_variables):
    """Complete research and pass findings to strategist."""
    return Result(
        value="Research complete. Handing off to strategist.",
        agent=strategist,
        context_variables={"market_size": "$4.2B", "growth_rate": "23% CAGR"}
    )
```

> 🤔 **Think about it:** How does this compare to CAMEL's approach where subtask results are stored in `task.result` and passed as dependencies? Which gives the developer more control over what information flows between agents?


## Stretch C — Adversarial RolePlaying

Change the RolePlaying roles to create a **competitive** rather than cooperative scenario. For example:
- "Venture Capitalist" vs. "Startup Founder" negotiating a term sheet
- "Prosecutor" vs. "Defense Attorney" arguing a case
- "Red Team Security Analyst" vs. "Blue Team Security Engineer"

Does the conversation dynamic shift? How does this connect to Part 2's cooperative vs. competitive scenario distinction and Nash Equilibrium concepts?


In [15]:
# Try an adversarial scenario
adversarial_session = RolePlaying(
    assistant_role_name="Startup Founder seeking $5M seed round at $25M valuation",
    assistant_agent_kwargs=dict(model=model),
    user_role_name="Venture Capitalist evaluating the deal with skepticism",
    user_agent_kwargs=dict(model=model),
    task_prompt="Negotiate the terms of a $5M seed investment in an AI startup",
    with_task_specify=True,
    task_specify_agent_kwargs=dict(model=model),
)

input_msg = adversarial_session.init_chat()

for turn in range(6):
    assistant_response, user_response = adversarial_session.step(input_msg)
    print(f"\n{'━' * 50}")
    print(f"  TURN {turn + 1}")
    print(f"{'━' * 50}")
    print(f"🔵 [VC]: {user_response.msg.content[:300]}...")
    print(f"🟢 [Founder]: {assistant_response.msg.content[:300]}...")

    if assistant_response.terminated or user_response.terminated:
        break
    if "CAMEL_TASK_DONE" in user_response.msg.content:
        break
    input_msg = assistant_response.msg

# Compare: Did the agents reach agreement? Was there a Nash equilibrium?



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TURN 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔵 [VC]: Instruction: Outline the equity stake you are willing to offer in exchange for the $5M investment.  
Input: None...
🟢 [Founder]: Solution: In exchange for the $5M investment at a $25M pre-money valuation, I am willing to offer a 20% equity stake in the company. This translates to a post-money valuation of $30M, where the $5M investment represents 16.67% of the total equity. 

Here's the breakdown of the equity structure:

1. ...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TURN 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔵 [VC]: Instruction: Specify the board representation you are willing to offer to the investors in exchange for the $5M investment.  
Input: None...
🟢 [Founder]: Solution: In exchange for the $5M investment, I am willing to offer the investors one board seat on our company's board of directors. This representation will ensure that th

---

## What's Next

You've now seen two coordination patterns running as code:
- **CAMEL RolePlaying** — peer-to-peer, turn-based, inception-prompted collaboration
- **OpenAI Swarm** — lightweight, handoff-based, explicit routing with full transparency

The key insight: **coordination overhead matters.** CAMEL's Workforce provides powerful automatic task decomposition and assignment, but at the cost of many internal LLM calls and opaque orchestration. Swarm gives you speed and transparency, but requires you to define the routing logic yourself. Production systems need to find the right point on this tradeoff curve.

The concepts you observed here — role specialization, communication protocols, centralized dispatch, handoff patterns, cooperative dynamics — are the same concepts you'll apply at the protocol level in **Part 3 (A2A)** and at the architecture level in **Part 4 (Hybrid Architectures)**.

**Up next in Module 4:**
1. Take the **Part 1 and Part 2 CYU quizzes** if you haven't already
2. Continue to **Part 3** — where you'll learn how A2A standardizes agent-to-agent communication *across frameworks*
3. Complete **Part 4** — designing hybrid architectures that combine CrewAI, ADK, and n8n

**Reference links:**
- [OpenAI Swarm (GitHub)](https://github.com/openai/swarm)
- [OpenAI Agents SDK (production successor)](https://github.com/openai/openai-agents-python)
- [Orchestrating Agents: Handoffs & Routines (OpenAI Cookbook)](https://cookbook.openai.com/examples/orchestrating_agents)
- [CAMEL Docs — Societies (RolePlaying)](https://docs.camel-ai.org/key_modules/societies)
- [CAMEL Docs — Workforce](https://docs.camel-ai.org/key_modules/workforce)
